In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from sklearn.metrics import mean_absolute_error, r2_score

In [7]:
# Load your dataset
import os
from pathlib import Path

data_path = Path.cwd().parent / "Dataset" / "set2_features.csv"
df = pd.read_csv(data_path)

# Separate features and target
X = df.drop(columns=["RUL(min)"])   # Features
y = df["RUL(min)"]                  # Target

In [8]:
print(df.head())

       mean       std       rms    max    min  peak_to_peak  skewness  \
0 -0.010196  0.073475  0.074179  0.454 -0.386         0.840  0.084000   
1 -0.002585  0.075338  0.075382  0.369 -0.388         0.757  0.052146   
2 -0.002484  0.076189  0.076230  0.503 -0.400         0.903  0.032810   
3 -0.002277  0.078691  0.078724  0.608 -0.576         1.184  0.041489   
4 -0.002404  0.078437  0.078474  0.391 -0.391         0.782  0.028226   

   kurtosis  crest_factor  shape_factor  impulse_factor  clearance_factor  \
0  0.629209      6.120331      1.271660        7.782979          9.258404   
1  0.648742      5.147086      1.277742        6.576647          7.834513   
2  0.513894      6.598472      1.265456        8.350077          9.907938   
3  1.158529      7.723217      1.281033        9.893696         11.794197   
4  0.603617      4.982524      1.278896        6.372127          7.622802   

     entropy  fft_mean   fft_std     fft_max      energy  RUL(min)  
0 -71.364266  7.553473  7.600

In [9]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [10]:
# Step 1: Random Forest for feature transformation
rf = RandomForestRegressor(n_estimators=300, random_state=42)

# Train RF
rf.fit(X_train, y_train)

# Transform features using RF (leaf indices)
X_train_rf = rf.apply(X_train)
X_test_rf = rf.apply(X_test)

In [11]:
scaler = StandardScaler()

# Scale RF output
X_train_rf = scaler.fit_transform(X_train_rf)
X_test_rf = scaler.transform(X_test_rf)

In [12]:
param_grid = {
    "C": [10, 50, 100],
    "gamma": ["scale", 0.01, 0.001],
    "epsilon": [0.1, 1, 10]
}

grid = GridSearchCV(
    SVR(kernel="rbf"),
    param_grid,
    cv=3,
    scoring="neg_mean_absolute_error",
    n_jobs=-1
)

grid.fit(X_train_rf, y_train)

svm = grid.best_estimator_

In [13]:
y_pred = svm.predict(X_test_rf)

print("MAE:", mean_absolute_error(y_test, y_pred))
print("R2 Score:", r2_score(y_test, y_pred))

MAE: 892.8572793867685
R2 Score: 0.7821258615377961


---

# Feature Extraction

In [14]:
def extract_features(signal):
    signal = np.array(signal)
    features = {}

    # basic
    mean = np.mean(signal)
    std = np.std(signal)
    rms = np.sqrt(np.mean(signal**2))
    peak = np.max(np.abs(signal))

    features["mean"] = mean
    features["std"] = std
    features["rms"] = rms
    features["max"] = np.max(signal)
    features["min"] = np.min(signal)
    features["peak_to_peak"] = np.ptp(signal)

    # statistical
    features["skewness"] = pd.Series(signal).skew()
    features["kurtosis"] = pd.Series(signal).kurt()

    # vibration specific
    mean_abs = np.mean(np.abs(signal))
    sqrt_mean = np.mean(np.sqrt(np.abs(signal)))

    features["crest_factor"] = peak / rms if rms != 0 else 0
    features["shape_factor"] = rms / mean_abs if mean_abs != 0 else 0
    features["impulse_factor"] = peak / mean_abs if mean_abs != 0 else 0
    features["clearance_factor"] = peak / (sqrt_mean**2) if sqrt_mean != 0 else 0

    # entropy
    hist, _ = np.histogram(signal, bins=50, density=True)
    hist = hist + 1e-12
    features["entropy"] = -np.sum(hist * np.log(hist))

    # frequency features
    fft_vals = np.abs(np.fft.fft(signal))
    fft_vals = fft_vals[:len(fft_vals)//2]

    features["fft_mean"] = np.mean(fft_vals)
    features["fft_std"] = np.std(fft_vals)
    features["fft_max"] = np.max(fft_vals)

    # energy
    features["energy"] = np.sum(signal**2)
    features["log_energy"] = np.sum(np.log(signal**2 + 1e-12))

    return pd.DataFrame([features])

In [15]:
def predict_rul_from_raw(signal):
    # Step 1: Extract features
    features = extract_features(signal)

    # Step 2: Align columns with training data
    features = features[X.columns]

    # Step 3: RF transformation
    features_rf = rf.apply(features)

    # Step 4: Scaling
    features_rf = scaler.transform(features_rf)

    # Step 5: Prediction
    rul = svm.predict(features_rf)

    return rul[0]

In [19]:
signal_path = Path.cwd().parent / "Test_signals" / "test_signal_2.txt"

if signal_path.is_dir():
    signal_files = sorted(
        p for p in signal_path.rglob("*")
        if p.is_file() and p.suffix.lower() in {".txt", ".csv", ".dat"}
    )
    if not signal_files:
        raise FileNotFoundError(f"No readable signal file found inside {signal_path}")
    signal_path = signal_files[0]

new_signal = np.loadtxt(signal_path, dtype=float, encoding="utf-8-sig")

predicted_rul = predict_rul_from_raw(new_signal)

print(f"Loaded {new_signal.size} samples from {signal_path.name}")
print("Predicted RUL in minutes:", predicted_rul)

Loaded 20480 samples from test_signal_2.txt
Predicted RUL in minutes: 3293.0622310921067


In [10]:
# Simulated raw vibration signal
new_signal = np.random.randn(1000)

predicted_rul = predict_rul_from_raw(new_signal)

print("Predicted RUL in minutes:", predicted_rul)

Predicted RUL in minutes: 1347.2744980555944


In [11]:
importance_df = pd.DataFrame({
    "Feature": X.columns,
    "Importance": rf.feature_importances_
}).sort_values(by="Importance", ascending=False)

print(importance_df)

             Feature  Importance
14           fft_std    0.509981
13          fft_mean    0.095518
2                rms    0.087651
1                std    0.087018
16            energy    0.078918
15           fft_max    0.024253
7           kurtosis    0.021004
9       shape_factor    0.018235
6           skewness    0.017470
0               mean    0.011706
12           entropy    0.010945
4                min    0.010642
3                max    0.007135
8       crest_factor    0.005677
10    impulse_factor    0.004945
5       peak_to_peak    0.004822
11  clearance_factor    0.004081
